## Dataset Preprations and Labelling

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir('/content/drive/Othercomputers/My Laptop/Work/Grievance Redressal Project')

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('data/grievance_dataset_keyword_extraction.csv')

# Expanded Vocabularies
high_keywords = [
  'urgent', 'emergency', 'danger', 'death', 'severe', 'immediate',
  'threat', 'suicide', 'bad condition', 'critical', 'fatal',
  'life-threatening', 'harassment', 'corruption', 'bribe', 'scam',
  'collapse', 'hazard', 'disaster', 'priority', 'accident'
]

medium_keywords = [
  'delay', 'pending', 'error', 'issue', 'not working', 'request',
  'status', 'update', 'complaint', 'refund', 'missing', 'broken',
  'failure', 'mistake', 'wait', 'stuck', 'processing', 'unhandled',
  'discrepancy', 'correction'
]

def assign_weak_label(text):
  text = str(text).lower()
  if any(word in text for word in high_keywords):
      return 'high'
  elif any(word in text for word in medium_keywords):
        return 'medium'
  else:
      return 'low'

df['urgency_label'] = df['grievance_text_processed'].apply(assign_weak_label)

Assigning labels based on expanded keywords...


In [ ]:
df.shape

(37854, 12)

In [ ]:
df.columns

Index(['_id', 'org_code', 'recvd_date', 'closing_date', 'sex', 'state',
       'grievance_text', 'remarks_text', 'grievance_text_processed',
       'yake_keywords', 'textrank_keywords', 'urgency_label'],
      dtype='object')

In [ ]:
print(df["urgency_label"].value_counts())

urgency_label
medium    23794
high       7870
low        6190
Name: count, dtype: int64


In [ ]:
df.to_csv('data/grievance_dataset_urgency_labels.csv')

## Train Multimodal Classifiers

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

def train_multimodal_classifiers():
    # 1. Load Data
    df = pd.read_csv('data/grievance_dataset_urgency_labels.csv')

    # Drop rows missing the target or critical text
    df = df.dropna(subset=['urgency_label', 'grievance_text_processed'])

    # 2. Feature Engineering (Extracting useful info from dates)
    df['recvd_date'] = pd.to_datetime(df['recvd_date'], errors='coerce')
    df['recvd_month'] = df['recvd_date'].dt.month.fillna(-1).astype(str)
    df['recvd_day'] = df['recvd_date'].dt.dayofweek.fillna(-1).astype(str)

    # Fill missing categorical values
    for col in ['org_code', 'sex', 'state']:
        df[col] = df[col].fillna('Unknown')

    # 3. Define Feature Columns
    text_feature = 'grievance_text_processed'
    categorical_features = ['org_code', 'sex', 'state', 'recvd_month', 'recvd_day']

    X = df[[text_feature] + categorical_features]
    y = df['urgency_label']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 4. Create the Column Transformer
    # This processes text and categorical data simultaneously
    preprocessor = ColumnTransformer(
        transformers=[
            ('text', TfidfVectorizer(max_features=10000, ngram_range=(1,2)), text_feature),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ]
    )

    # 5. Define Models to Test
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
        "Linear SVC": LinearSVC(class_weight='balanced', dual=False, max_iter=2000),
        "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    }

    # 6. Train and Evaluate Each Model
    for name, model in models.items():
        print(f"--- Training {name} ---")

        # Bundle preprocessing and modeling code in a pipeline
        clf_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])

        # Train
        clf_pipeline.fit(X_train, y_train)

        # Predict
        predictions = clf_pipeline.predict(X_test)

        # Evaluate
        print(classification_report(y_test, predictions, zero_division=0))
        print("\n")

if __name__ == "__main__":
    train_multimodal_classifiers()

--- Training Logistic Regression ---
              precision    recall  f1-score   support

        high       0.89      0.87      0.88      1551
         low       0.70      0.95      0.81      1181
      medium       0.96      0.89      0.93      4839

    accuracy                           0.90      7571
   macro avg       0.85      0.90      0.87      7571
weighted avg       0.91      0.90      0.90      7571



--- Training Linear SVC ---
              precision    recall  f1-score   support

        high       0.97      0.90      0.93      1551
         low       0.85      0.95      0.90      1181
      medium       0.96      0.96      0.96      4839

    accuracy                           0.94      7571
   macro avg       0.93      0.93      0.93      7571
weighted avg       0.95      0.94      0.94      7571



--- Training Random Forest ---
              precision    recall  f1-score   support

        high       0.99      0.72      0.83      1551
         low       0.94      

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import xgboost as xgb

def train_gpu_classifier():
    # 1. Load Data
    df = pd.read_csv('data/grievance_dataset_urgency_labels.csv')
    df = df.dropna(subset=['urgency_label', 'grievance_text_processed'])

    # 2. Feature Engineering
    df['recvd_date'] = pd.to_datetime(df['recvd_date'], errors='coerce')
    df['recvd_month'] = df['recvd_date'].dt.month.fillna(-1).astype(str)
    df['recvd_day'] = df['recvd_date'].dt.dayofweek.fillna(-1).astype(str)

    for col in ['org_code', 'sex', 'state']:
        df[col] = df[col].fillna('Unknown')

    # 3. Define Features and Target
    text_feature = 'grievance_text_processed'
    categorical_features = ['org_code', 'sex', 'state', 'recvd_month', 'recvd_day']

    X = df[[text_feature] + categorical_features]

    le = LabelEncoder()
    y = le.fit_transform(df['urgency_label'])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 4. Create the Column Transformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('text', TfidfVectorizer(max_features=10000, ngram_range=(1,2)), text_feature),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ]
    )

    # 5. Define GPU-Accelerated Model (Updated syntax)
    gpu_model = xgb.XGBClassifier(
        tree_method='hist',
        device='cuda',
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42
    )

    # 6. Build Pipeline and Train
    print("Training XGBoost on GPU...")
    clf_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', gpu_model)
    ])

    clf_pipeline.fit(X_train, y_train)

    # 7. Predict and Evaluate
    predictions = clf_pipeline.predict(X_test)

    print("\nModel Evaluation:")
    target_names = le.inverse_transform([0, 1, 2])
    print(classification_report(y_test, predictions, target_names=target_names, zero_division=0))

if __name__ == "__main__":
    train_gpu_classifier()

Training XGBoost on GPU...

Model Evaluation:
              precision    recall  f1-score   support

        high       0.27      0.63      0.38      1551
         low       0.16      0.03      0.06      1181
      medium       0.74      0.56      0.64      4839

    accuracy                           0.50      7571
   macro avg       0.39      0.41      0.36      7571
weighted avg       0.55      0.50      0.50      7571



/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [09:41:21] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


## Saving the best model

In [ ]:
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.svm import LinearSVC

# 1. Load Data
df = pd.read_csv('data/grievance_dataset_urgency_labels.csv')
df = df.dropna(subset=['urgency_label', 'grievance_text_processed'])

# 2. Feature Engineering
df['recvd_date'] = pd.to_datetime(df['recvd_date'], errors='coerce')
df['recvd_month'] = df['recvd_date'].dt.month.fillna(-1).astype(str)
df['recvd_day'] = df['recvd_date'].dt.dayofweek.fillna(-1).astype(str)

for col in ['org_code', 'sex', 'state']:
    df[col] = df[col].fillna('Unknown')

# 3. Define Features
text_feature = 'grievance_text_processed'
categorical_features = ['org_code', 'sex', 'state', 'recvd_month', 'recvd_day']

X = df[[text_feature] + categorical_features]
y = df['urgency_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Create the Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=10000, ngram_range=(1,2)), text_feature),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# 5. Build and Train the Winning Pipeline
print("Training Final Linear SVC Model...")
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LinearSVC(class_weight='balanced', dual=False, max_iter=2000))
])

final_pipeline.fit(X_train, y_train)

# 6. Final Evaluation
print("\nFinal Model Evaluation on Test Data:")
predictions = final_pipeline.predict(X_test)
print(classification_report(y_test, predictions, zero_division=0))


Training Final Linear SVC Model...

Final Model Evaluation on Test Data:
              precision    recall  f1-score   support

        high       0.97      0.90      0.93      1551
         low       0.85      0.95      0.90      1181
      medium       0.96      0.96      0.96      4839

    accuracy                           0.94      7571
   macro avg       0.93      0.93      0.93      7571
weighted avg       0.95      0.94      0.94      7571



In [56]:
import pickle

# Save the SVM model
model_path = '/content/drive/Othercomputers/My Laptop/Work/Grievance Redressal Project/05-Severity-Risk-Scoring/model/urgency_model_svc.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(final_pipeline, f)

print(f"Model saved successfully to '{model_path}'")

Model saved successfully to '/content/drive/Othercomputers/My Laptop/Work/Grievance Redressal Project/05-Severity-Risk-Scoring/model/urgency_model_svc.pkl'
